In [20]:
import pandas as pd
import numpy as np

In [21]:
# Genre dataset (you already used)
genre_df = pd.read_csv("data_w_genres.csv")

# Song dataset
song_df = pd.read_csv("data.csv")

print("Genre columns:", genre_df.columns)
print("Song columns:", song_df.columns)

Genre columns: Index(['genres', 'artists', 'acousticness', 'danceability', 'duration_ms',
       'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness',
       'tempo', 'valence', 'popularity', 'key', 'mode', 'count'],
      dtype='str')
Song columns: Index(['valence', 'year', 'acousticness', 'artists', 'danceability',
       'duration_ms', 'energy', 'explicit', 'id', 'instrumentalness', 'key',
       'liveness', 'loudness', 'mode', 'name', 'popularity', 'release_date',
       'speechiness', 'tempo'],
      dtype='str')


In [22]:
genre_df = genre_df[['genres', 'valence', 'energy']].dropna()

genre_df.head()

,genres,valence,energy
0,['show tunes'],0.389500,0.394003
1,[],0.268865,0.406808
2,[],0.354857,0.286571
3,[],0.372030,0.245770
4,[],0.482286,0.488286


In [23]:
emotion_features = {
    "Happy": {"valence": 0.9, "energy": 0.8, "danceability": 0.75},
    "Sad": {"valence": 0.2, "energy": 0.3, "danceability": 0.35},
    "Angry": {"valence": 0.3, "energy": 0.9, "danceability": 0.55},
    "Fear": {"valence": 0.2, "energy": 0.4, "danceability": 0.30},
    "Surprise": {"valence": 0.7, "energy": 0.8, "danceability": 0.70},
    "Neutral": {"valence": 0.5, "energy": 0.5, "danceability": 0.50}
}

In [24]:
def recommend_genres(emotion, top_n=5):
    
    target = emotion_features[emotion]
    
    genre_df['distance'] = np.sqrt(
        (genre_df['valence'] - target['valence'])**2 +
        (genre_df['energy'] - target['energy'])**2
    )
    
    return genre_df.sort_values('distance').head(top_n)['genres'].tolist()

In [25]:
# Check columns
print(song_df.columns)

Index(['valence', 'year', 'acousticness', 'artists', 'danceability',
       'duration_ms', 'energy', 'explicit', 'id', 'instrumentalness', 'key',
       'liveness', 'loudness', 'mode', 'name', 'popularity', 'release_date',
       'speechiness', 'tempo'],
      dtype='str')


In [26]:
song_df = song_df.dropna()

song_df.head()

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667,0.211,0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10,0.665,-20.096,1,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4,1921,0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,180533,0.341,0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7,0.160,-12.441,1,Clancy Lowered the Boom,5,1921,0.4150,60.936
2,0.0394,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062,0.166,0,1o6I8BglA6ylDMrIELygv1,0.913000,3,0.101,-14.850,1,Gati Bali,5,1921,0.0339,110.339
3,0.1650,1921,0.967,['Frank Parker'],0.275,210000,0.309,0,3ftBPsC5vPBKxYSee08FDH,0.000028,5,0.381,-9.316,1,Danny Boy,3,1921,0.0354,100.109
4,0.2530,1921,0.957,['Phil Regan'],0.418,166693,0.193,0,4d6HGyGT8e121BsdKmw9v6,0.000002,3,0.229,-10.096,1,When Irish Eyes Are Smiling,2,1921,0.0380,101.665


In [27]:
def get_songs_from_emotion(emotion, top_n=5):
    target = emotion_features[emotion]
    
    scored = song_df.copy()
    scored['distance'] = np.sqrt(
        ((scored['valence'] - target['valence']) * 1.4) ** 2 +
        ((scored['energy'] - target['energy']) * 1.2) ** 2 +
        ((scored['danceability'] - target['danceability']) * 1.0) ** 2
    )
    
    scored['ranking_score'] = scored['distance'] - (scored['popularity'] / 1000.0)
    ranked = scored.sort_values(['ranking_score', 'popularity'], ascending=[True, False])
    
    return ranked['name'].drop_duplicates().head(top_n).tolist()

In [28]:
def recommend_music(emotion):
    
    # Step 1: Get genres
    genres = recommend_genres(emotion)
    
    # Step 2: Get songs
    songs = get_songs_from_emotion(emotion)
    
    return genres, songs

In [29]:
emotion = "Happy"

genres, songs = recommend_music(emotion)

print("Emotion:", emotion)
print("Genres:", genres)
print("Songs:", songs)

Emotion: Happy
Genres: ["['latin', 'tropical']", '[]', "['dance pop', 'pop', 'pop dance', 'pop rock', 'post-teen pop', 'talent show', 'tropical house', 'uk pop']", '[]', "['tejano', 'tex-mex']"]
Songs: ['We Own The Night', 'Uneasy Hearts Weigh The Most']
